# Trees and Search

| Difficulty | ███░░░░░░░ 3/10 |
| :--- | :--- |
| Prerequisites | 07.02 |
| Time | ~60 min |
| Colab GPU | Not needed |

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NiravRVaghasiya/maths-for-ai/blob/main/07_Discrete_Mathematics/03_trees_and_search.ipynb)

---

## 🎯 Learning Objective

Learn tree structures, BFS/DFS traversal, and the trie data structure for efficient prefix search.

---

## 📐 Theory

A **tree** is a connected graph (`07.02`) with no cycles — equivalently, a graph on $n$
vertices with exactly $n-1$ edges where every vertex is reachable from every other. Trees are
the natural shape for anything hierarchical: file systems, parse structures, decision
procedures, and (as this notebook's AI section shows directly) the branching search space of
sequence generation.

### Rooted trees and vocabulary

A **rooted tree** designates one vertex as the **root**; every other vertex then has a unique
**parent** (the neighbor closer to the root) and zero or more **children**. A vertex with no
children is a **leaf**. The **depth** of a vertex is its distance from the root, and the
**height** of the tree is the maximum depth over all vertices. Rooted trees inherit everything
from graph theory (`07.02`) but add a canonical direction — "down" toward leaves — that plain
graphs don't have.

### Traversal: BFS and DFS

Visiting every vertex of a tree (or graph) systematically means picking an order, and the two
canonical orders reveal complementary structure:

- **Breadth-first search (BFS)** visits vertices level by level: the root, then all its
  children, then all their children, and so on, using a **queue** (first-in-first-out). BFS
  finds the *shortest path* (fewest edges) from the start vertex to any other vertex, because it
  never visits a vertex at distance $d+1$ before finishing every vertex at distance $d$.
- **Depth-first search (DFS)** plunges down one branch as far as possible before backtracking,
  using a **stack** (last-in-first-out, often implemented via recursion). DFS doesn't guarantee
  shortest paths, but it's often cheaper in memory (it only needs to remember one path from root
  to current vertex, not an entire frontier) and it naturally enumerates things like "every leaf,
  in left-to-right order."

Both algorithms visit every vertex and edge at most once, giving $O(|V|+|E|)$ time — the two
differ only in *order*, not in which vertices get visited.

### Dynamic programming on trees (and beyond)

**Dynamic programming (DP)** solves a problem by breaking it into overlapping subproblems,
solving each *exactly once*, and reusing (memoizing) the result rather than recomputing it. Two
ingredients make DP applicable: **optimal substructure** (an optimal solution is built from
optimal solutions to subproblems) and **overlapping subproblems** (the same subproblem recurs
many times if solved naively). On a tree, computing something about a subtree (its height, its
sum, its best-scoring path) only needs already-computed results from its children — no subtree
is ever revisited once solved. This is the same "solve small pieces once, reuse everywhere" idea
behind Pascal's identity (`07.01`); DP is the general technique for turning exponential brute
force into polynomial (or linear) time by never solving the same subproblem twice.

### Tries: trees specialized for prefixes

A **trie** (prefix tree) is a rooted tree purpose-built for storing strings: each edge is
labeled by one character, and the string spelled out by the path from the root to a vertex is a
**prefix** shared by every string stored beneath that vertex. Two or more words sharing a
prefix (`under`, `understand`, `understanding`) share the trie nodes for that prefix and only
branch where the words actually differ. This makes checking "does any stored word start with
this prefix?" cost proportional to the *prefix length*, never to how many words are stored —
independent of vocabulary size, unlike scanning a flat list of strings one at a time.

---

In [ ]:
# Setup
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt
from collections import deque
%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid')

## 👁️ Visual Intuition

Drawing the same tree with BFS and DFS visit-order labels side by side shows the level-by-level
vs. branch-by-branch difference directly. We use the same small tree throughout this notebook
so every section builds on one concrete, visualizable example.

In [ ]:
# The same small tree, visited two ways: BFS goes level-by-level, DFS goes branch-by-branch.
tree_edges = [("A", "B"), ("A", "C"), ("B", "D"), ("B", "E"), ("C", "F"), ("C", "G"), ("E", "H")]
T = nx.Graph()
T.add_edges_from(tree_edges)

def bfs_order(G, root):
    visited, order, queue = {root}, [root], deque([root])
    while queue:
        node = queue.popleft()
        for neighbor in sorted(G[node]):  # sorted() only for a reproducible, deterministic order
            if neighbor not in visited:
                visited.add(neighbor)
                order.append(neighbor)
                queue.append(neighbor)
    return order

def dfs_order(G, root):
    visited, order = set(), []
    def recurse(node):
        visited.add(node)
        order.append(node)
        for neighbor in sorted(G[node]):
            if neighbor not in visited:
                recurse(neighbor)
    recurse(root)
    return order

bfs_sequence = bfs_order(T, "A")
dfs_sequence = dfs_order(T, "A")
bfs_rank = {node: i for i, node in enumerate(bfs_sequence)}
dfs_rank = {node: i for i, node in enumerate(dfs_sequence)}

pos = nx.bfs_layout(T, "A")
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for ax, rank, title in zip(axes, [bfs_rank, dfs_rank],
                            ["BFS visit order (level by level)", "DFS visit order (branch by branch)"]):
    labels = {n: f"{n}\n#{rank[n]}" for n in T.nodes()}
    nx.draw(T, pos, ax=ax, with_labels=True, labels=labels,
             node_color="#4C72B0", font_color="white", node_size=900)
    ax.set_title(title)
plt.tight_layout()
plt.show()

print(f"BFS visit order: {bfs_sequence}")
print(f"DFS visit order: {dfs_sequence}")
print(f"Both visit all {T.number_of_nodes()} vertices exactly once -- only the ORDER differs.")

## 🐍 Implementation from Scratch

We implement BFS and DFS from scratch and cross-check them against `networkx`, build a trie
from scratch and confirm it finds prefixes in time independent of vocabulary size, and
implement dynamic programming on a grid-path counting problem to make the "overlapping
subproblems" idea from the Theory section directly visible in the call counts.

In [ ]:
class TrieNode:
    """One node per shared character; is_end marks 'a complete word ends here'."""
    def __init__(self):
        self.children = {}
        self.is_end = False

class Trie:
    def __init__(self):
        self.root = TrieNode()
        self.n_nodes = 1  # count nodes created, to measure how much structure is actually shared

    def insert(self, word):
        node = self.root
        for ch in word:
            if ch not in node.children:
                node.children[ch] = TrieNode()
                self.n_nodes += 1
            node = node.children[ch]
        node.is_end = True

    def search(self, word):
        """Exact match: does this EXACT word exist in the trie?"""
        node = self.root
        for ch in word:
            if ch not in node.children:
                return False
            node = node.children[ch]
        return node.is_end

    def starts_with(self, prefix):
        """Does ANY stored word start with this prefix? Cost = O(len(prefix)), not O(vocab size)."""
        node = self.root
        for ch in prefix:
            if ch not in node.children:
                return False
            node = node.children[ch]
        return True

    def longest_prefix_match(self, text, start=0):
        """Longest stored word that is a prefix of text[start:] -- the core operation a
        greedy tokenizer performs at every position. Returns (end_index, steps_taken)."""
        node, longest_end, steps, i = self.root, -1, 0, start
        while i < len(text) and text[i] in node.children:
            node = node.children[text[i]]
            i += 1
            steps += 1
            if node.is_end:
                longest_end = i
        return longest_end, steps

vocabulary = ["play", "player", "playing", "plays", "plan", "planet", "plant",
              "cat", "car", "care", "careful"]
trie = Trie()
for word in vocabulary:
    trie.insert(word)

flat_char_count = sum(len(w) for w in vocabulary)
print(f"Trie nodes for {len(vocabulary)} words: {trie.n_nodes} "
      f"(vs. {flat_char_count} characters if stored as separate flat strings)")
print(f"search('play')={trie.search('play')}, search('pla')={trie.search('pla')}, "
      f"starts_with('pla')={trie.starts_with('pla')}")
assert trie.search("play") and not trie.search("pla") and trie.starts_with("pla")

longest_end, steps = trie.longest_prefix_match("playing")
print(f"longest_prefix_match('playing') -> matched {'playing'[:longest_end]!r} "
      f"in {steps} trie steps")
assert "playing"[:longest_end] == "playing"  # "playing" itself is in the vocabulary

# --- BFS/DFS from scratch, cross-checked against networkx ---
def bfs(adjacency, start):
    visited, order, queue = {start}, [], deque([start])
    while queue:
        node = queue.popleft()
        order.append(node)
        for neighbor in sorted(adjacency[node]):
            if neighbor not in visited:
                visited.add(neighbor)
                queue.append(neighbor)
    return order

def dfs(adjacency, start):
    visited, order = set(), []
    def recurse(node):
        visited.add(node)
        order.append(node)
        for neighbor in sorted(adjacency[node]):
            if neighbor not in visited:
                recurse(neighbor)
    recurse(start)
    return order

G_check = nx.Graph([("A", "B"), ("A", "C"), ("B", "D"), ("B", "E"), ("C", "F"), ("E", "G")])
adjacency = {n: list(G_check.neighbors(n)) for n in G_check.nodes()}
our_bfs, our_dfs = bfs(adjacency, "A"), dfs(adjacency, "A")
nx_bfs = list(nx.bfs_tree(G_check, "A").nodes())
nx_dfs = list(nx.dfs_tree(G_check, "A").nodes())
print(f"\nOur BFS:      {our_bfs}")
print(f"networkx BFS: {nx_bfs}  (match: {our_bfs == nx_bfs})")
print(f"Our DFS:      {our_dfs}")
print(f"networkx DFS: {nx_dfs}  (match: {our_dfs == nx_dfs})")
assert our_bfs == nx_bfs and our_dfs == nx_dfs

# --- Dynamic programming: counting grid paths, to make "overlapping subproblems" concrete ---
def count_paths_naive(r, c, call_counter):
    call_counter[0] += 1
    if r == 0 or c == 0:
        return 1
    return count_paths_naive(r - 1, c, call_counter) + count_paths_naive(r, c - 1, call_counter)

def count_paths_memoized(r, c, cache, call_counter):
    call_counter[0] += 1
    if (r, c) in cache:
        return cache[(r, c)]  # the ENTIRE point of DP: never recompute a solved subproblem
    result = 1 if (r == 0 or c == 0) else (
        count_paths_memoized(r - 1, c, cache, call_counter) +
        count_paths_memoized(r, c - 1, cache, call_counter)
    )
    cache[(r, c)] = result
    return result

print(f"\n{'(rows,cols)':>12} {'naive calls':>12} {'memoized calls':>16} {'answer':>10}")
for r, c in [(6, 6), (10, 10), (12, 12)]:
    naive_counter = [0]
    naive_answer = count_paths_naive(r, c, naive_counter)
    memo_counter = [0]
    memo_answer = count_paths_memoized(r, c, {}, memo_counter)
    assert naive_answer == memo_answer
    print(f"{f'({r},{c})':>12} {naive_counter[0]:>12,} {memo_counter[0]:>16,} {naive_answer:>10,}")
print("Same answer every time -- memoization changes only HOW MANY subproblems get solved,")
print("collapsing exponential re-computation into one call per distinct (r,c) pair.")

## 🤖 Why This Matters for AI

Two direct uses: subword tokenizers (BPE, WordPiece, SentencePiece) store their vocabulary in a
trie-like structure so that finding the longest matching subword prefix at each position costs
time proportional to that subword's length, not to how many tens of thousands of tokens the
vocabulary contains. Separately, **beam search** — the decoding strategy every practical
autoregressive language model uses — is literally a **bounded-width breadth-first search**
over the tree of possible token sequences: BFS explores an entire level (every child of every
current node) before moving deeper, and beam search does exactly that but keeps only the
top-$k$ scoring nodes at each level rather than all of them, trading search completeness for
tractability (recall `07.01`'s combinatorial-explosion argument for why "all of them" stops
being an option almost immediately).

In [ ]:
# --- Part 1: tries make tokenizer-style prefix lookup independent of vocabulary size ---
rng = np.random.default_rng(5)  # fresh RNG, self-contained for this cell
alphabet = np.array(list("abcdefghijklmnopqrstuvwxyz"))

def random_word(rng, length):
    return "".join(rng.choice(alphabet, size=length))

def naive_longest_prefix_match(text, vocab_list, start=0):
    """What you'd do WITHOUT a trie: check every vocabulary entry one at a time."""
    longest_end, comparisons = -1, 0
    for word in vocab_list:
        comparisons += 1
        if text.startswith(word, start) and (longest_end == -1 or start + len(word) > longest_end):
            longest_end = start + len(word)
    return longest_end, comparisons

base_vocab = ["under", "understand", "understanding", "stand", "standing",
              "ing", "er", "the", "th", "read", "reading", "re", "un"]
filler_tokens = [random_word(rng, int(rng.integers(3, 10))) for _ in range(5000)]
vocab_large = base_vocab + filler_tokens  # a much bigger vocabulary, real subwords unaffected

text = "understanding"
print(f"{'vocabulary':>14} {'size':>6} {'trie steps':>11} {'naive comparisons':>19}")
results = {}
for vocab_list, label in [(base_vocab, "small"), (vocab_large, "large")]:
    trie = Trie()
    for word in vocab_list:
        trie.insert(word)
    _, trie_steps = trie.longest_prefix_match(text)
    _, naive_comparisons = naive_longest_prefix_match(text, vocab_list)
    results[label] = (len(vocab_list), trie_steps, naive_comparisons)
    print(f"{label:>14} {len(vocab_list):>6} {trie_steps:>11} {naive_comparisons:>19}")

vocab_growth = results["large"][0] / results["small"][0]
print(f"\nGrowing the vocabulary {vocab_growth:.0f}x left the trie's step count UNCHANGED "
      f"({results['small'][1]} -> {results['large'][1]}), while the naive scan's comparison "
      f"count grew in lock-step with vocabulary size ({results['small'][2]} -> {results['large'][2]}).")
print("This is exactly why real BPE/WordPiece tokenizers use trie-like structures internally.")

# --- Part 2: beam search IS bounded-width BFS over the tree of token sequences ---
toy_vocab = ["a", "b", "c"]  # tiny so the FULL, unpruned BFS tree stays small enough to show
seq_len = 4

def deterministic_string_seed(s):
    """A reproducible string hash. Python's builtin hash() is randomized per-process
    (PYTHONHASHSEED), so it would silently change every re-run -- this doesn't."""
    h = 0
    for ch in s:
        h = (h * 131 + ord(ch)) % (2**31 - 1)
    return h

def sequence_score(seq, seed=0):
    """Deterministic pseudo-random score standing in for a language model's cumulative log-prob."""
    local_rng = np.random.default_rng((deterministic_string_seed(seq) ^ seed) % (2**31))
    return local_rng.normal()

def full_bfs_over_sequences(vocab, length):
    """Ordinary BFS: expand EVERY node at the current level before going deeper. No pruning."""
    frontier = deque([("", 0.0)])
    level_sizes = []
    for _ in range(length):
        next_frontier = deque()
        while frontier:
            seq, cum_score = frontier.popleft()
            for token in vocab:
                next_frontier.append((seq + token, cum_score + sequence_score(seq + token)))
        level_sizes.append(len(next_frontier))
        frontier = next_frontier
    return level_sizes

def beam_search_over_sequences(vocab, length, beam_width):
    """Beam search: THE SAME level-by-level BFS expansion, but prune to the top beam_width
    scoring nodes after each level -- bounding the frontier width instead of letting it grow."""
    frontier = [("", 0.0)]
    candidates_per_level = []
    for _ in range(length):
        candidates = [(seq + token, cum_score + sequence_score(seq + token))
                       for seq, cum_score in frontier for token in vocab]
        candidates_per_level.append(len(candidates))
        candidates.sort(key=lambda pair: -pair[1])
        frontier = candidates[:beam_width]
    return candidates_per_level, frontier

bfs_level_sizes = full_bfs_over_sequences(toy_vocab, seq_len)
print(f"\nFull BFS (vocab size={len(toy_vocab)}, depth={seq_len}) frontier size per level: "
      f"{bfs_level_sizes}, total nodes visited: {sum(bfs_level_sizes)}")

for k in [1, 2]:
    beam_level_sizes, final_beam = beam_search_over_sequences(toy_vocab, seq_len, beam_width=k)
    print(f"Beam search k={k}: candidates scored per level: {beam_level_sizes}, "
          f"total: {sum(beam_level_sizes)}, best sequence found: {final_beam[0][0]!r}")
print("\nBeam search visits a small, CONSTANT-width slice of the same tree full BFS explores --")
print("the exact same 'keep top-k, discard the rest' idea behind pruned search everywhere.")

## 🏋️ Exercises

Six levels, in order: mechanical computation → conceptual understanding → derivation →
implementation → application → open-ended challenge. Solutions/outlines are in the collapsed
`Solution` blocks below each exercise — try before revealing.

### 🔢 COMPUTE
1. By hand, write out the BFS and DFS visit orders starting from vertex `B` (not the usual root
   `A`) for this notebook's own tree (`tree_edges`). As a structural sanity check, confirm your
   tree has exactly $n-1=7$ edges for its $n=8$ vertices (the defining property of a tree from
   the Theory section), then verify both orders using the `bfs`/`dfs` functions.

<details>
<summary>💡 Solution</summary>

Starting from `B`, BFS visits level by level ($B$ itself, then $B$'s neighbors, then their
unvisited neighbors, ...): $B\to A\to D\to E\to C\to H\to F\to G$. DFS plunges down one branch
first (following the same sorted-neighbor tie-breaking the `bfs`/`dfs` functions use):
$B\to A\to C\to F\to G\to D\to E\to H$. Both visit all $8$ vertices exactly once, confirmed by
`bfs_order(T,"B")` and `dfs_order(T,"B")`. Structural check: `tree_edges` has $7$ entries, and
$n-1=8-1=7$ ✓ — consistent with `T` being a tree (a connected graph with no cycles has exactly
$n-1$ edges).

</details>

### 💭 UNDERSTAND
2. BFS's memory cost is dominated by its frontier (queue) width; DFS's is dominated by its
   maximum recursion/stack depth. Without running any code, predict which search uses
   *more peak memory* on (a) a tree that is a single root with 20 direct children (wide,
   shallow) and (b) a tree that is a single chain of 20 vertices (narrow, deep) — then verify
   by tracking `len(queue)` for BFS and recursion depth for DFS on both shapes.

<details>
<summary>💡 Solution</summary>

(a) Wide, shallow tree: BFS's frontier must hold all 20 children simultaneously once it reaches
that level, so its peak memory is $\Theta(20)$; DFS never has more than 2 vertices on its call
stack at once (the root, then whichever single child it's currently visiting), so DFS uses far
less memory here. (b) Narrow, deep chain: BFS's frontier never holds more than 1 vertex at a
time (each vertex has exactly one unvisited neighbor to enqueue), so BFS's peak memory is
$O(1)$; DFS's call stack grows one frame per vertex on the chain, reaching depth 21 — DFS uses
far more memory here. Verifying: wide tree gives BFS max frontier $20$ vs. DFS max stack depth
$2$; narrow chain gives BFS max frontier $1$ vs. DFS max stack depth $21$ — confirming the
prediction flips depending on tree *shape*, not on which algorithm is "generally" more
memory-efficient.

</details>

### ✏️ DERIVE
3. The grid-path count from $(0,0)$ to $(r,c)$ (moving only right or down) satisfies the
   recurrence $\text{count}(r,c)=\text{count}(r-1,c)+\text{count}(r,c-1)$ with base case
   $\text{count}(r,0)=\text{count}(0,c)=1$ — exactly `count_paths_memoized`'s recursion. Derive
   the closed form $\text{count}(r,c)=\binom{r+c}{c}$ by recognizing this as a special case of a
   result from `07.01`, and confirm numerically that `count_paths_memoized(10,10,{},[0])` equals
   `comb(20,10,exact=True)`.

<details>
<summary>💡 Solution outline</summary>

A path from $(0,0)$ to $(r,c)$ using only right/down moves is a sequence of exactly $r+c$ moves,
of which $r$ are "down" and $c$ are "right" (in some order) — any such sequence uniquely
determines, and is uniquely determined by, the path. Counting these sequences is exactly
`07.01`'s multinomial-coefficient setup with 2 labeled groups (down-moves and right-moves) out
of $r+c$ total slots: $\binom{r+c}{r,c}=\binom{r+c}{r}=\binom{r+c}{c}$ (using the
$\binom{n}{k}=\binom{n}{n-k}$ symmetry from `07.01`, since choosing which $r$ of the $r+c$ moves
are "down" is equivalent to choosing which $c$ are "right"). This closed form satisfies the same
recurrence as Pascal's identity ($\binom{n}{k}=\binom{n-1}{k-1}+\binom{n-1}{k}$, with
$n=r+c,k=c$: moving down means one fewer "down" move remains, i.e. $\binom{(r-1)+c}{c}$; moving
right means one fewer "right" move remains, i.e. $\binom{r+(c-1)}{c-1}$) which is exactly
$\text{count}(r,c)=\text{count}(r-1,c)+\text{count}(r,c-1)$ restated in binomial notation —
confirming the recursion and the closed form describe the same function. Numerically:
`count_paths_memoized(10,10,{},[0])=184756` and `comb(20,10,exact=True)=184756` — match.

</details>

### 🐍 IMPLEMENT
4. Add a `delete(word)` method to the `Trie` class that removes a word by unmarking its
   `is_end` flag (removing now-unused nodes is optional but a good extra challenge).

<details>
<summary>✅ How to know you're right</summary>

After `trie.delete("plan")` on the Implementation section's `vocabulary`, three things must all
hold simultaneously: `trie.search("plan")` becomes `False` (the exact word is gone),
`trie.search("planet")` stays `True` (a longer word sharing the "plan" prefix is untouched,
since deletion only unmarks `is_end` rather than removing the shared nodes those other words
still need), and `trie.starts_with("pla")` stays `True` (the prefix itself is still reachable
via `plan`'s own nodes, which remain in the tree en route to `planet`/`plant`/`play`/etc. even
though `plan` itself is no longer a complete stored word).

</details>

### 🤖 APPLY
5. A dependency parse represents a sentence as a tree: each word is a node, and each edge
   connects a word to the word it grammatically depends on. Build the tree for "The cat sat on
   the mat" with edges `sat-cat, cat-The, sat-on, on-mat, mat-the` (root `sat`), then use BFS
   distance (number of edges on the shortest path) to compute how many dependency "hops" apart
   `"The"` and `"the"` are — a question the shortest linear (word-position) distance between
   them would answer completely differently.

<details>
<summary>✅ What you should observe</summary>

BFS distance from `"The"` to `"the"` should come out to **5** hops
(`The`→`cat`→`sat`→`on`→`mat`→`the`), even though the two words are close together in raw
sentence position (word 1 and word 5 out of 6, only 4 positions apart) — the point is that
*syntactic* distance (tree hops) and *linear* distance (word position) measure genuinely
different things, and the dependency tree captures the former, which is often the more relevant
one for tasks like coreference or long-range agreement. Sanity check: BFS distance should be
symmetric (`dist(a,b)==dist(b,a)`, since the underlying graph is undirected) — confirm this for
at least one other pair, e.g. `"cat"` and `"on"` (distance 2, via `sat`).

</details>

### 🚀 CHALLENGE
6. The grid-path DP in the Implementation section only *counts* paths. Modify it to compute the
   *maximum-scoring* path instead: assign a random score to each grid cell, and have
   `count_paths_memoized`'s recursion take the max (not the sum) of the two directions it could
   have arrived from, plus the current cell's score. Separately, implement a "greedy" path
   choice that at each step picks whichever of the two next cells (right or down) has the higher
   *immediate* score, with no lookahead. Compare the greedy total to the true DP optimum, and
   connect your finding to why beam search with beam width $k=1$ is exactly this kind of greedy,
   locally-optimal search, and why it can fail to find the true best-scoring full sequence the
   way exhaustive DP (or a wide-enough beam) would.

<details>
<summary>🔍 What a strong answer covers</summary>

A strong answer runs both and shows a concrete gap: on one random $5\times5$ score grid, greedy
found a total score of about $3.62$ against the true DP optimum of about $5.79$ — a real,
non-trivial gap, not a near-miss — because greedy's "pick the better immediate neighbor" rule
has no way to know that the *locally* worse move now might open onto a run of high-scoring cells
later, while DP correctly accounts for the entire remaining path from every cell via the
recursion. This is structurally identical to why beam-width-1 search (equivalent to greedy
decoding) can miss a higher-scoring full sequence that requires accepting a lower-probability
token at some step to reach a much better continuation afterward — both are the same "greedy
now can lose to a globally-planned alternative" failure mode, and both are fixed the same way in
principle (exhaustive search / DP considers every path; widening the beam is a practical
middle ground that recovers some, but not all, of DP's guarantee without paying DP's full cost
on an exponentially large sequence space where exhaustive DP itself becomes infeasible).

</details>

---

## 📚 Further Reading
- Cormen, Leiserson, Rivest & Stein, *Introduction to Algorithms*, Ch. 22 (Elementary Graph Algorithms) and Ch. 15 (Dynamic Programming)
- Sennrich, Haddow & Birch, ["Neural Machine Translation of Rare Words with Subword Units"](https://arxiv.org/abs/1508.07909) (BPE tokenization)
- Freitag & Al-Onaizan, ["Beam Search Strategies for Neural Machine Translation"](https://arxiv.org/abs/1702.01806)

---

*Next notebook: [Automata and Formal Languages](04_automata_and_formal_languages.ipynb)*